# PI-10 UAT
This notebook addresses requirements in []

In [ ]:
import geopandas as gpd
import pandas as pd
import xarray as xr

In [ ]:
nhf ="../data/nhf_1.2.2.gpkg"

## Reservoirs - NWM v3 RFC and USACE

In [ ]:
res_index = "../data/sconus/lakes/input/reservoir_index_AnA.nc"
active_rfc = "../data/sconus/lakes/input/nwps_all_gauges_report.csv"

res_da = gpd.read_file(nhf, layer="reservoir_da")
lakes = gpd.read_file(nhf, layer="lakes")
gages = gpd.read_file(nhf, layer="gages")
ds_res_index = xr.open_dataset(res_index)
df_active_rfc = pd.read_csv(active_rfc)

usace_gage_id_field: str = "usace_gage_id"
usace_lake_id_field: str = "usace_lake_id"
rfc_gage_id_field: str = "rfc_gage_id"
rfc_lake_id_field: str = "rfc_lake_id"
active_gage_id: str = "nws shef id"

In [ ]:
# rfc
def get_crosswalk(ds: xr.Dataset, gage_field, lake_field, output_gage_field: str = 'site_no', lake_id_field: str = 'lake_id', active_rfc: pd.DataFrame | None = None):
    crosswalk = pd.DataFrame(
                data={
                    gage_field: ds[gage_field].to_numpy(),
                    lake_field: ds[lake_field].to_numpy(),
                }
            )
    crosswalk[gage_field] = crosswalk[gage_field].apply(lambda x: x.decode("utf-8")).str.strip()


    # for RFC gages, filter by active gages if the data is available
    if gage_field == rfc_gage_id_field and active_rfc is not None:
        crosswalk = crosswalk.loc[crosswalk[gage_field].isin(active_rfc[active_gage_id])].copy()

    crosswalk.rename(columns={gage_field: output_gage_field, lake_field: lake_id_field}, inplace=True)

    return crosswalk

def assert_gage_present(res_type: str, crosswalk: pd.DataFrame, gages: gpd.GeoDataFrame, gage_id_field: str ='site_no'):
    missing_gages = crosswalk.loc[~crosswalk[gage_id_field].astype(str).isin(gages[gage_id_field])].copy()
    len_missing = len(missing_gages)
    print(f"{len_missing} {res_type} missing gages in gage layer")
    return missing_gages

def assert_lake_present(res_type: str, crosswalk: pd.DataFrame, lakes: gpd.GeoDataFrame, lake_id_field: str ='lake_id'):
    missing_lakes = crosswalk.loc[~crosswalk[lake_id_field].astype(str).isin(lakes[lake_id_field])].copy()
    len_missing = len(missing_lakes)
    print(f"{len_missing} {res_type} missing lakes in lakes layer")
    return missing_lakes



            


In [ ]:
crosswalk_usace = get_crosswalk(ds_res_index, usace_gage_id_field, usace_lake_id_field)

In [ ]:
assert_gage_present('USACE', crosswalk_usace,  gages)

In [ ]:
assert_lake_present('USACE', crosswalk_usace, lakes)

In [ ]:
crosswalk_rfc = get_crosswalk(ds_res_index, rfc_gage_id_field, rfc_lake_id_field, active_rfc=df_active_rfc)

In [ ]:
assert_gage_present('RFC', crosswalk_rfc, gages)

In [ ]:
assert_lake_present('RFC', crosswalk_rfc, lakes)